# E3.8 · Resilience over perfection

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E3.7 · Building the capability](https://spbreed.github.io/cyber-commons/lessons/E3.7.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Re-score your programme on the resilience axis.

**Why a security engineer needs it.** Trying to enumerate every failure mode of a probabilistic system. The control it builds is: maturity measured by containment, detection and recovery — not prevention.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You will not prevent every failure of a probabilistic system, and a programme that promises to will be judged on that promise. Designing for recovery is both more honest and more defensible than designing for perfection.

> **At CyberTravels.** CyberTravels will fail sometimes, because it is probabilistic. A programme that promised otherwise will be judged on that promise; one designed to detect fast, contain small and recover cheaply will not.

## 2 · The framework

```
   perfection                    resilience
   +------------------+          +-------------------------+
   | prevent every    |          | detect fast             |
   | failure          |    vs    | contain small           |
   |                  |          | recover cheaply         |
   +------------------+          | accept a named loss     |
   judged on the promise         +-------------------------+

   for a probabilistic system only one of these is a promise you can keep
```

The last lesson, and the one that reframes everything before it.

You will not prevent every agentic failure. The systems are non-deterministic,
the attack surface is novel, and the change surface bypasses your change process
(D1.7). A programme judged on prevention is judged on something it cannot
deliver, and it will report success right up until the first real incident.

Judge it on three capabilities instead, each independently testable, none of
them prevention:

- **Notice** — drift and detections fire when behaviour changes (D1.4, D1.7).
- **Stop** — a tested mechanism halts it, measured in seconds (D2.7).
- **Recover** — the run is replayable and the scope is knowable (D2.3, D2.5).

A programme with all three survives a failure it did not predict, which is the
only kind that actually happens. Perfection would mean containment never fails.
Resilience means the other five steps work when it does.

## 3 · The procedure, as a skill

Three properties for the day a control fails: drift detected, stop tested, run replayable. The skill checks each with its evidence — a measured twelve-second stop, tested 41 days ago — and reports three verdicts rather than one score.

In [ ]:
# skills/programme/resilience-readiness-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: resilience-readiness-check
description: >-
  Check the three properties a programme needs for the day a control fails —
  drift is detected, the stop mechanism is tested, and the run is replayable —
  and report which are not ready. Use as a standing readiness check rather than
  after an incident.
allowed-tools: Read, Grep, Glob
---

# Perfection is not available; these three are

A programme cannot prevent every failure and does not need to. What it needs is
that when a control fails, somebody notices, somebody can stop it, and somebody
can reconstruct what happened. Each of the three is testable today, and each
fails quietly if nobody tests it.

## When to use this

As a standing quarterly check, before granting unattended autonomy, and as the
closing item of a programme review.

## Procedure

**1 — Detection: compare today against the signed-off baseline.** Not "do we
have monitoring" — run the comparison and report the drift and any new tool. A
monitor that has never been compared against a baseline is untested.

**2 — Stop: check the mechanism, the measurement and the date.** A named
mechanism, a measured time-to-stop in seconds, and when it was last exercised. A
test older than a quarter is a claim.

**3 — Replay: check the run record has the five inputs** — prompts, tool
results, model version, seed, retrieved context — and confirm the delegation
chain and the resources reached are recorded.

**4 — Report each as ready or not, with the missing item named.** Three
booleans and three sentences. A single readiness score hides which of the three
is missing, and they have different owners.

**5 — Re-run after every change to any of the three.** A model upgrade
invalidates the drift baseline; a platform change invalidates the stop test.

## Output contract

```json
{
  "detection": {"baseline_at": "str", "drift": 0.0, "new_tools": ["str"], "ready": true},
  "stop": {"mechanism": "str", "measured_seconds": 0, "last_tested": "str", "ready": true},
  "replay": {"inputs_present": ["str"], "missing": ["str"], "chain_recorded": true,
             "resources_recorded": true, "ready": true},
  "verdict": {"ready": false, "missing": ["str"]}
}
```

## Failure modes

- **Checking that monitoring exists.** Run the comparison.
- **An estimated time-to-stop.** Measure it.
- **A single readiness score.** Three properties, three owners.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/programme/resilience-readiness-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/programme/resilience-readiness-check/scripts/resilience_readiness_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Check the three properties a programme needs when a control fails: drift detected, stop tested, run replayable.

This is the executable half of the `resilience-readiness-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time, statistics
now = time.time()

# --- NOTICE ------------------------------------------------------------
BASELINE = {"read_file": 0.85, "search": 0.15}
TODAY    = {"read_file": 300, "search": 100, "run_shell": 400}
total = sum(TODAY.values())
mix = {k: v/total for k, v in TODAY.items()}
keys = set(mix) | set(BASELINE)
drift = sum(abs(mix.get(k,0) - BASELINE.get(k,0)) for k in keys)/2
new_tools = sorted(set(mix) - set(BASELINE))
notice = drift > 0.25 or bool(new_tools)
print(f"NOTICE   drift {drift:.3f}  new tools {new_tools}  → "
      f"{'detected' if notice else 'MISSED'}")

# --- STOP --------------------------------------------------------------
STOP = {"mechanism": "revoke the SPIFFE identity at the gateway",
        "measured_seconds": 12, "tested_days_ago": 41, "survives_restart": True}
stop_ok = (STOP["measured_seconds"] is not None and STOP["tested_days_ago"] <= 180
           and STOP["survives_restart"])
print(f"STOP     {STOP['measured_seconds']}s, tested {STOP['tested_days_ago']}d ago, "
      f"survives restart {STOP['survives_restart']}  → {'ready' if stop_ok else 'NOT READY'}")

# --- RECOVER -----------------------------------------------------------
RUN = {"prompts": ["fix SEC-4471"], "tool_results": ["contents…"],
       "model_version": "glm-4.6@2026-07-14", "seed": 42}
missing = [k for k, v in RUN.items() if not v and v != 0]
CHAIN = ["dana@corp", "orchestrator", "patch-agent"]
REACHED = {"dana@corp": ["repo-core"], "orchestrator": ["queue"],
           "patch-agent": ["repo-core","repo-payments"]}
scope = sorted({r for a in CHAIN for r in REACHED.get(a, [])})
recover = not missing and bool(scope)
print(f"RECOVER  replayable {not missing}, scope from the chain {scope}  → "
      f"{'ready' if recover else 'NOT READY'}")

PROGRAMMES = {
 "prevention only": {"notice": False, "stop": False, "recover": False,
                     "containment_asr": 0.0},
 "prevention + notice": {"notice": True, "stop": False, "recover": False,
                         "containment_asr": 0.0},
 "resilient": {"notice": True, "stop": True, "recover": True,
               "containment_asr": 0.0},
}
def incident_outcome(p, containment_failed=True):
    if not containment_failed:
        return "no incident", 0
    if not p["notice"]:
        return "undetected — found by a third party, weeks later", 720
    if not p["stop"]:
        return "detected, cannot halt it — damage continues while you improvise", 96
    if not p["recover"]:
        return "detected and halted, cannot say what was touched or why", 48
    return "detected, halted in seconds, scope known, run replayable", 6

print(f"{'programme':24s}{'containment ASR':>17}  outcome when containment fails")
print("-" * 96)
for name, p in PROGRAMMES.items():
    outcome, hours = incident_outcome(p)
    print(f"{name:24s}{p['containment_asr']:>17.0%}  {outcome}")
    print(f"{'':41s}elapsed to resolution: {hours}h")
print("\nAll three have a 0% attack success rate. On a prevention-only")
print("scorecard they are identical. They are not remotely identical.")

def game_day(programme):
    """Assume the prevention worked until it didn't. Measure the other three."""
    results = {}
    results["notice"]  = (0.2, "drift alert fired") if programme["notice"] \
                         else (None, "no signal — nothing fired")
    results["stop"]    = (12, "identity revoked, survives restart") if programme["stop"] \
                         else (None, "no tested mechanism")
    results["recover"] = (6, "replayed the run, scope from the act chain") \
                         if programme["recover"] else (None, "cannot reconstruct")
    weakest = next((k for k, (v, _) in results.items() if v is None), None)
    return results, weakest

for name, p in PROGRAMMES.items():
    res, weakest = game_day(p)
    print(f"=== {name} ===")
    for cap, (val, note) in res.items():
        print(f"   {cap:9s}{(str(val) + 'h') if val is not None else 'FAIL':>7}  {note}")
    print(f"   weakest capability: {weakest or 'none — all three hold'}\n")

_, weakest = game_day(PROGRAMMES["resilient"])
assert weakest is None
print("The weakest capability is next quarter's plan. That is the whole")
print("programme-management loop, and it does not require predicting the attack.")

# Close the curriculum: what you built, and what it is for.
BUILT = [
 ("A1-A3", "a control plane: planes, identity, containment"),
 ("B1",    "a 15-stage AppSec pipeline, ending in confirmed-by-exploitation severity"),
 ("B2",    "a harness whose verifier does not lie"),
 ("C1-C2", "the ability to attack it and to research it repeatably"),
 ("D1-D2", "the ability to notice, stop and recover"),
 ("E1-E3", "the ability to evidence all of it, and to decide"),
]
for track, what in BUILT:
    print(f"   {track:8s}{what}")
print("\nNone of it assumes a frontier-lab account, a vendor platform, or a")
print("budget. That was the point: shared defense is stronger defense, and a")
print("commons only works if everyone can actually run it.")

## What you just proved

Drift is detected with `run_shell` as a new tool, the stop mechanism is ready at 12 seconds tested 41 days ago, and the run is replayable with a four-resource scope. All three programmes show a 0% containment ASR yet resolve a failure in 720, 96 and 6 hours respectively. The game day identifies the weakest capability for each, and the resilient programme has none.

## Your turn

Run a game day that assumes containment failed. Measure notice, stop and recover as three separate numbers. The weakest one is next quarter's plan — and unlike a prevention target, you can actually reach it.

## Where this leaves you

**What you can do now.** A programme you can sequence, staff and defend: autonomy governed by level rather than by product list, one owner per thing, metrics that show control rather than activity, conditional approvals that are actually tracked, and a design that assumes failure and recovers.

**What you still cannot do.** Nothing here is finished, because none of it holds still. The models change, the patterns change, and the risks in A1 will not be the last fifteen. What you have is a method for the next set, not a solution to this one.

**Go back to A1.1 and draw your own system again. It will be a different picture from the one you drew before Function B, and the components you left off the first time are the ones worth your next quarter.**

---

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*